In [ ]:
import os
import xir
from vart import RunnerExt
import numpy as np
import matplotlib.pyplot as plt
import time

In [ ]:
model_file = "outputs2d/compile_out/vxm2d.xmodel"
graph= xir.Graph.deserialize(model_file)
root= graph.get_root_subgraph()

def collect_subgraphs(sg):
    out = []
    for child in sg.get_children():
        out.append(child)
        out.extend(collect_subgraphs(child))
    return out

subgraphs = collect_subgraphs(root)
print(f"Found {len(subgraphs)} subgraphs:")
for idx, sg in enumerate(subgraphs):
    device = sg.get_attr("device") if sg.has_attr("device") else "CPU"
    print(f"[{idx:02d}] {sg.get_name()} → {device}")

In [ ]:
runners = [RunnerExt.create_runner(sg, "run") for sg in subgraphs]
print("\nRunners created:", len(runners))

in_bufs, out_bufs = runners[0].get_inputs(), runners[-1].get_outputs()
print("First runner input shapes :", [b.get_tensor().get_shape() for b in in_bufs])
print("Last runner output shapes  :", [b.get_tensor().get_shape() for b in out_bufs])

In [ ]:
data = np.load("calibration_dataset/sample0.npy")
moving = data["moving"].astype(np.float32)
fixed = data["fixed"].astype(np.float32)
batch = [moving[None, ...], fixed[None, ...]]
print("\nInitial batch shapes:", [b.shape for b in batch])

In [ ]:
start_all = time.time()
current = batch

for sg, runner in zip(subgraphs, runners):
    ins, outs = runner.get_inputs(), runner.get_outputs()
    for buf, arr in zip(ins, current):
        buf.copy_from_host(arr)
    job = runner.execute_async(ins, outs)
    runner.wait(job)
    current = [np.asarray(buf)[0] for buf in outs]
    print(f"{sg.get_name():40s} → output {current[0].shape}")

total_ms = (time.time() - start_all) * 1000
print(f"\nTotal latency (ms): {total_ms:.2f}")

warped = current[0][..., 0]

In [ ]:
fig, axes = plt.subplots(1, 3,figsize=(12, 4))
axes[0].imshow(batch[0][0, ..., 0],cmap="gray"); axes[0].set_title("Moving")
axes[1].imshow(batch[1][0, ..., 0],cmap="gray"); axes[1].set_title("Fixed")
axes[2].imshow(warped,cmap="gray"); axes[2].set_title("Warped")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()